# Immunotherapy Response Prediction
This notebook demonstrates the complete reproducible workflow. It uses **synthetic data**, so the displayed performance is not biological or clinical evidence. Replace the demo-loading cell with validated cohort data for a real study.

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
from src.immunotherapy_model import make_demo
from src.full_analysis import run_full_analysis
print(f'Project root: {ROOT}')

## 1. Create a reproducible demonstration cohort
The generator creates an imbalanced binary classification problem that mimics a high-dimensional response study without representing real patients or real genes.

In [ ]:
X, y, sample_ids = make_demo(seed=42)
print('Expression shape:', X.shape)
print('Class counts:')
print(y.value_counts().rename(index={0: 'Non-response', 1: 'Response'}))
X.head()

## 2. Run the complete analysis
Preprocessing is learned inside each cross-validation fold. Model selection uses training data only, and final metrics come from an untouched test split.

In [ ]:
output_dir = ROOT / 'results' / 'notebook_demo'
summary = run_full_analysis(X, y, sample_ids, output_dir, top_k=50, seed=42, demo=True)
summary['selected_model'], summary['held_out_test'][summary['selected_model']]

## 3. Inspect model comparison and important genes

In [ ]:
import pandas as pd
cv = pd.read_csv(output_dir / 'tables' / 'cross_validation_results.csv')
importance = pd.read_csv(output_dir / 'tables' / 'permutation_importance.csv')
display(cv)
display(importance.head(15))

## 4. Review generated figures

In [ ]:
from IPython.display import Image, display
for name in ['pca_by_response.png', 'model_comparison.png', 'roc_curves.png', 'precision_recall_curves.png', 'top_gene_importance.png']:
    display(Image(filename=str(output_dir / 'figures' / name)))

## Interpretation checklist
- Prefer cross-validation means and variability over a single accuracy value.
- Examine average precision when responders are uncommon.
- Treat gene importance as predictive association, not causation.
- Check patient grouping, treatment definitions, and batch effects before analyzing real cohorts.
- Require external validation before making biomarker claims.